# 03 Superellipse Sanity Gate

This notebook is a pre-dataset gate for the superellipse family (new main nontrivial target).

Goals:
- inspect lattice shape changes with exponent `n`,
- check whether low levels `E0..E3` versus `n` are smooth enough,
- monitor site-count jumps `N_sites(n)` as a discretization-artifact indicator.

No dataset generation and no ML are done here.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse.linalg import eigsh

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.geometry import build_superellipse_dot

In [ ]:
def first_four_energies_superellipse(a: float, b: float, n: float) -> np.ndarray:
    """Compute first four levels for one superellipse geometry."""
    fsys = build_superellipse_dot(a=a, b=b, n=n)
    h = fsys.hamiltonian_submatrix(sparse=True).tocsr()
    dim = h.shape[0]
    if dim < 5:
        vals = np.linalg.eigvalsh(h.toarray())[: min(4, dim)]
    else:
        vals, _ = eigsh(h, k=4, which="SA")
    vals = np.sort(np.asarray(vals, dtype=float))
    if vals.shape[0] != 4:
        raise ValueError("Expected 4 energies; increase geometry size.")
    return vals

In [ ]:
# Visualize representative lattice shapes.
a_shape, b_shape = 10.0, 8.0
n_rep = [1.2, 1.5, 2.0, 3.0, 4.0, 6.0]

fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey=True)
for ax, n_val in zip(axes.ravel(), n_rep):
    fsys = build_superellipse_dot(a=a_shape, b=b_shape, n=n_val)
    xy = np.array([site.pos for site in fsys.sites], dtype=float)
    ax.scatter(xy[:, 0], xy[:, 1], s=8)
    ax.set_title(f"n={n_val}, N={len(fsys.sites)}")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.suptitle("Superellipse lattice sites for representative n")
plt.tight_layout()
plt.show()

In [ ]:
# One example system and its first 4 energies.
a_ex, b_ex, n_ex = 10.0, 8.0, 2.0
fsys_ex = build_superellipse_dot(a=a_ex, b=b_ex, n=n_ex)
energies_ex = first_four_energies_superellipse(a=a_ex, b=b_ex, n=n_ex)

print(f"Example parameters: a={a_ex}, b={b_ex}, n={n_ex}")
print(f"N_sites = {len(fsys_ex.sites)}")
for i, e in enumerate(energies_ex):
    print(f"E{i} = {e:.10f}")

In [ ]:
def run_n_sweep(a: float, b: float, n_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return energies and site counts for a sweep over n."""
    energies = []
    n_sites = []
    for n_val in n_values:
        fsys = build_superellipse_dot(a=a, b=b, n=float(n_val))
        n_sites.append(len(fsys.sites))
        energies.append(first_four_energies_superellipse(a=a, b=b, n=float(n_val)))
    return np.array(energies), np.array(n_sites, dtype=int)

In [ ]:
def n_jump_locations(n_values: np.ndarray, n_sites: np.ndarray) -> np.ndarray:
    """Return n positions where site count changes."""
    mask = np.diff(n_sites) != 0
    return n_values[1:][mask]

In [ ]:
# Coarse sweep values (required representative points).
n_coarse = np.array([1.2, 1.5, 2.0, 3.0, 4.0, 6.0], dtype=float)

# Symmetric and anisotropic cases.
a_sym, b_sym = 10.0, 10.0
a_aniso, b_aniso = 12.0, 8.0

E_sym_c, N_sym_c = run_n_sweep(a=a_sym, b=b_sym, n_values=n_coarse)
E_an_c, N_an_c = run_n_sweep(a=a_aniso, b=b_aniso, n_values=n_coarse)

print("Coarse sweep complete.")

In [ ]:
# Fine sweep only in main working range n in [1.2, 4.0].
n_fine = np.linspace(1.2, 4.0, 57)

E_sym_f, N_sym_f = run_n_sweep(a=a_sym, b=b_sym, n_values=n_fine)
E_an_f, N_an_f = run_n_sweep(a=a_aniso, b=b_aniso, n_values=n_fine)

jumps_sym = n_jump_locations(n_fine, N_sym_f)
jumps_an = n_jump_locations(n_fine, N_an_f)

print(f"Fine sweep symmetric: {len(jumps_sym)} N_sites jumps in [1.2, 4.0]")
print(f"Fine sweep anisotropic: {len(jumps_an)} N_sites jumps in [1.2, 4.0]")

In [ ]:
def plot_sweep(n_values: np.ndarray, energies: np.ndarray, n_sites: np.ndarray, jumps: np.ndarray, title_prefix: str) -> None:
    """Plot E0..E3 and N_sites versus n with N_sites jump markers."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    ax_e, ax_n = axes
    for i in range(4):
        ax_e.plot(n_values, energies[:, i], marker="o", ms=3, label=f"E{i}")
    for nj in jumps:
        ax_e.axvline(nj, color="gray", lw=0.8, alpha=0.5)
    ax_e.set_xlabel("n")
    ax_e.set_ylabel("Energy")
    ax_e.set_title(f"{title_prefix}: E0..E3 vs n")
    ax_e.grid(True, alpha=0.3)
    ax_e.legend()

    ax_n.plot(n_values, n_sites, marker="o", ms=3, color="black")
    for nj in jumps:
        ax_n.axvline(nj, color="gray", lw=0.8, alpha=0.5)
    ax_n.set_xlabel("n")
    ax_n.set_ylabel("N_sites")
    ax_n.set_title(f"{title_prefix}: N_sites vs n")
    ax_n.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_sweep(n_values=n_coarse, energies=E_sym_c, n_sites=N_sym_c, jumps=n_jump_locations(n_coarse, N_sym_c), title_prefix="Coarse symmetric (a=b=10)")
plot_sweep(n_values=n_coarse, energies=E_an_c, n_sites=N_an_c, jumps=n_jump_locations(n_coarse, N_an_c), title_prefix="Coarse anisotropic (a=12, b=8)")

In [ ]:
plot_sweep(n_values=n_fine, energies=E_sym_f, n_sites=N_sym_f, jumps=jumps_sym, title_prefix="Fine symmetric (a=b=10)")
plot_sweep(n_values=n_fine, energies=E_an_f, n_sites=N_an_f, jumps=jumps_an, title_prefix="Fine anisotropic (a=12, b=8)")

In [ ]:
def summarize_e0_jumps(n_values: np.ndarray, e0: np.ndarray, n_sites: np.ndarray) -> dict[str, float]:
    """Compute compact E0/N_sites jump diagnostics for gate decisions."""
    e0_range = float(np.max(e0) - np.min(e0))
    if e0.size < 2:
        largest_local_jump = 0.0
    else:
        largest_local_jump = float(np.max(np.abs(np.diff(e0))))
    ratio = largest_local_jump / e0_range if e0_range > 0 else np.nan
    return {
        "n_jumps": float(np.sum(np.diff(n_sites) != 0)),
        "e0_range": e0_range,
        "largest_local_jump": largest_local_jump,
        "jump_to_range_ratio": ratio,
    }

In [ ]:
# Larger-scale fine sweep extension (same n-range for direct comparison).
n_fine_large = np.linspace(1.2, 4.0, 57)

# Requested larger-scale cases.
a_sym_l, b_sym_l = 20.0, 20.0
a_an_l, b_an_l = 24.0, 16.0

E_sym_l, N_sym_l = run_n_sweep(a=a_sym_l, b=b_sym_l, n_values=n_fine_large)
E_an_l, N_an_l = run_n_sweep(a=a_an_l, b=b_an_l, n_values=n_fine_large)

jumps_sym_l = n_jump_locations(n_fine_large, N_sym_l)
jumps_an_l = n_jump_locations(n_fine_large, N_an_l)

print(f"Large fine symmetric: {len(jumps_sym_l)} N_sites jumps in [1.2, 4.0]")
print(f"Large fine anisotropic: {len(jumps_an_l)} N_sites jumps in [1.2, 4.0]")

In [ ]:
plot_sweep(
    n_values=n_fine_large,
    energies=E_sym_l,
    n_sites=N_sym_l,
    jumps=jumps_sym_l,
    title_prefix="Fine large symmetric (a=b=20)",
)
plot_sweep(
    n_values=n_fine_large,
    energies=E_an_l,
    n_sites=N_an_l,
    jumps=jumps_an_l,
    title_prefix="Fine large anisotropic (a=24, b=16)",
)

In [ ]:
sym_summary_l = summarize_e0_jumps(n_values=n_fine_large, e0=E_sym_l[:, 0], n_sites=N_sym_l)
an_summary_l = summarize_e0_jumps(n_values=n_fine_large, e0=E_an_l[:, 0], n_sites=N_an_l)

print("Large-scale fine sweep summary")
print("- Symmetric (a=b=20):")
print(f"  N_sites jumps: {int(sym_summary_l['n_jumps'])}")
print(f"  E0 total range: {sym_summary_l['e0_range']:.6e}")
print(f"  Largest local E0 jump: {sym_summary_l['largest_local_jump']:.6e}")
print(f"  Largest-jump / total-range ratio: {sym_summary_l['jump_to_range_ratio']:.6f}")

print("- Anisotropic (a=24, b=16):")
print(f"  N_sites jumps: {int(an_summary_l['n_jumps'])}")
print(f"  E0 total range: {an_summary_l['e0_range']:.6e}")
print(f"  Largest local E0 jump: {an_summary_l['largest_local_jump']:.6e}")
print(f"  Largest-jump / total-range ratio: {an_summary_l['jump_to_range_ratio']:.6f}")

In [ ]:
def summarize_level_jumps(level_values: np.ndarray, n_sites: np.ndarray) -> dict[str, float]:
    """Jump/range diagnostics for one energy level across n."""
    level_range = float(np.max(level_values) - np.min(level_values))
    largest_local_jump = float(np.max(np.abs(np.diff(level_values)))) if level_values.size > 1 else 0.0
    ratio = largest_local_jump / level_range if level_range > 0 else np.nan
    return {
        "n_jumps": float(np.sum(np.diff(n_sites) != 0)),
        "range": level_range,
        "largest_jump": largest_local_jump,
        "jump_to_range_ratio": ratio,
    }

In [ ]:
# Final extra-large fine sweep extension.
n_fine_xl = np.linspace(1.2, 4.0, 57)

a_sym_xl, b_sym_xl = 30.0, 30.0
a_an_xl, b_an_xl = 36.0, 24.0

E_sym_xl, N_sym_xl = run_n_sweep(a=a_sym_xl, b=b_sym_xl, n_values=n_fine_xl)
E_an_xl, N_an_xl = run_n_sweep(a=a_an_xl, b=b_an_xl, n_values=n_fine_xl)

jumps_sym_xl = n_jump_locations(n_fine_xl, N_sym_xl)
jumps_an_xl = n_jump_locations(n_fine_xl, N_an_xl)

print(f"Extra-large fine symmetric: {len(jumps_sym_xl)} N_sites jumps in [1.2, 4.0]")
print(f"Extra-large fine anisotropic: {len(jumps_an_xl)} N_sites jumps in [1.2, 4.0]")

In [ ]:
plot_sweep(
    n_values=n_fine_xl,
    energies=E_sym_xl,
    n_sites=N_sym_xl,
    jumps=jumps_sym_xl,
    title_prefix="Fine extra-large symmetric (a=b=30)",
)
plot_sweep(
    n_values=n_fine_xl,
    energies=E_an_xl,
    n_sites=N_an_xl,
    jumps=jumps_an_xl,
    title_prefix="Fine extra-large anisotropic (a=36, b=24)",
)

In [ ]:
sym_summary_xl_e0 = summarize_level_jumps(E_sym_xl[:, 0], N_sym_xl)
an_summary_xl_e0 = summarize_level_jumps(E_an_xl[:, 0], N_an_xl)
sym_summary_xl_e3 = summarize_level_jumps(E_sym_xl[:, 3], N_sym_xl)
an_summary_xl_e3 = summarize_level_jumps(E_an_xl[:, 3], N_an_xl)

print("Extra-large fine sweep summary")
print("- Symmetric (a=b=30):")
print(f"  N_sites jumps: {int(sym_summary_xl_e0['n_jumps'])}")
print(f"  E0 total range: {sym_summary_xl_e0['range']:.6e}")
print(f"  Largest local E0 jump: {sym_summary_xl_e0['largest_jump']:.6e}")
print(f"  E0 largest-jump / total-range ratio: {sym_summary_xl_e0['jump_to_range_ratio']:.6f}")
print(f"  E3 largest-jump / total-range ratio (optional): {sym_summary_xl_e3['jump_to_range_ratio']:.6f}")

print("- Anisotropic (a=36, b=24):")
print(f"  N_sites jumps: {int(an_summary_xl_e0['n_jumps'])}")
print(f"  E0 total range: {an_summary_xl_e0['range']:.6e}")
print(f"  Largest local E0 jump: {an_summary_xl_e0['largest_jump']:.6e}")
print(f"  E0 largest-jump / total-range ratio: {an_summary_xl_e0['jump_to_range_ratio']:.6f}")
print(f"  E3 largest-jump / total-range ratio (optional): {an_summary_xl_e3['jump_to_range_ratio']:.6f}")

In [ ]:
# Compact cross-scale comparison for gate decision.
sym_small_e0 = summarize_level_jumps(E_sym_f[:, 0], N_sym_f)
sym_large_e0 = summarize_level_jumps(E_sym_l[:, 0], N_sym_l)
sym_xl_e0 = summarize_level_jumps(E_sym_xl[:, 0], N_sym_xl)

an_small_e0 = summarize_level_jumps(E_an_f[:, 0], N_an_f)
an_large_e0 = summarize_level_jumps(E_an_l[:, 0], N_an_l)
an_xl_e0 = summarize_level_jumps(E_an_xl[:, 0], N_an_xl)

print("Cross-scale summary (E0 jump/range metric)")
print("Scale        | Sym jumps | Sym ratio | Aniso jumps | Aniso ratio")
print("-------------+-----------+-----------+-------------+------------")
print(f"small        | {int(sym_small_e0['n_jumps']):9d} | {sym_small_e0['jump_to_range_ratio']:.6f} | {int(an_small_e0['n_jumps']):11d} | {an_small_e0['jump_to_range_ratio']:.6f}")
print(f"large        | {int(sym_large_e0['n_jumps']):9d} | {sym_large_e0['jump_to_range_ratio']:.6f} | {int(an_large_e0['n_jumps']):11d} | {an_large_e0['jump_to_range_ratio']:.6f}")
print(f"extra-large  | {int(sym_xl_e0['n_jumps']):9d} | {sym_xl_e0['jump_to_range_ratio']:.6f} | {int(an_xl_e0['n_jumps']):11d} | {an_xl_e0['jump_to_range_ratio']:.6f}")